In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: Uma Função Qiskit da Global Data Quantum


*Consulte a [referência da API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** As Funções Qiskit são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em status de pré-lançamento e sujeitas a alterações.
## Visão geral
O Quantum Portfolio Optimizer é uma Função Qiskit que aborda o problema de otimização dinâmica de portfólio, um problema padrão em finanças que visa rebalancear investimentos periódicos em um conjunto de ativos, maximizando retornos e minimizando riscos. Ao empregar técnicas de otimização quântica de ponta, esta função simplifica o processo para que usuários, sem expertise em computação quântica, possam se beneficiar de suas vantagens na busca de trajetórias de investimento ideais. Ideal para gestores de portfólio, pesquisadores em finanças quantitativas e investidores individuais, esta ferramenta permite o back-testing de estratégias de negociação em otimização de portfólio.
## Descrição da função
A função Quantum Portfolio Optimizer usa o algoritmo Variational Quantum Eigensolver (VQE) para resolver um problema de Otimização Binária Irrestrita Quadrática (QUBO), abordando problemas de otimização dinâmica de portfólio. Os usuários precisam apenas fornecer os dados de preço dos ativos e definir a restrição de investimento; em seguida, a função executa o processo de otimização quântica que retorna um conjunto de trajetórias de investimento otimizadas.

O processo consiste em quatro etapas principais. Primeiro, os dados de entrada são mapeados para um problema compatível com quantum, construindo o QUBO do problema de otimização dinâmica de portfólio e transformando-o em um operador quântico (Hamiltoniano de Ising). Em seguida, o problema de entrada e o algoritmo VQE são adaptados para execução no hardware quântico. O algoritmo VQE é então executado no hardware quântico e, por fim, os resultados são pós-processados para fornecer as trajetórias de investimento ideais. O sistema também inclui um pós-processamento ciente de ruído (baseado em [SQD](/guides/qiskit-addons-sqd)) para maximizar a qualidade da saída.

Esta Função Qiskit é baseada no [artigo publicado](https://arxiv.org/abs/2412.19150) pela Global Data Quantum.
![Visualização do fluxo de trabalho da função](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Como começar
Autentique-se usando sua [chave de API](https://quantum.cloud.ibm.com/) e selecione a Função Qiskit conforme descrito a seguir. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

Para este exemplo, usamos os ativos [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) e [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). A figura a seguir ilustra os dados usados neste exemplo, mostrando a evolução diária do preço de fechamento dos ativos de 1º de janeiro a 1º de setembro de 2023.

Neste exemplo, para garantir uniformidade entre as datas, preenchemos os dias sem negociação com o preço de fechamento da data disponível anterior. Aplicamos esta etapa porque os ativos selecionados provêm de diferentes mercados com dias de negociação variados, tornando essencial padronizar o conjunto de dados para consistência.
![Visualização dos dados históricos dos ativos](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Defina o problema.
Defina as especificações do problema configurando os parâmetros no dicionário `qubo_settings`.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

### 3. Defina as configurações do otimizador e do ansatz (Opcional)
Opcionalmente, defina requisitos específicos para o processo de otimização, incluindo a seleção do otimizador e seus parâmetros, bem como a especificação da primitiva e suas configurações.

Para o Ansatz Tailored, o tamanho de população escolhido foi baseado em experimentos anteriores que mostraram que este valor produz uma otimização estável e eficiente.

No caso do Ansatz Real Amplitudes, você pode seguir uma relação linear entre o ``population_size`` e o número de qubits no circuito. Como regra geral aproximada, é recomendado usar um ``population_size`` **mínimo** de `~ 0.8 * n_qubits` para o ansatz `real_amplitudes`.

Espera-se que o Optimized Real Amplitudes tenha um desempenho de otimização melhor do que o ansatz Real Amplitudes. No entanto, o número de variáveis a otimizar neste ansatz aumenta muito mais rapidamente do que no caso do Real Amplitudes (veja o [artigo](https://arxiv.org/pdf/2412.19150)). Portanto, para problemas grandes, o Optimized Real Amplitudes requer mais execuções de circuito. O Optimized Real Amplitudes provavelmente será útil para problemas que requerem até 100 qubits, mas é recomendado ter cuidado ao definir os parâmetros ``population_size``. Como exemplo desse aumento de escala em ``population_size``, a tabela anterior mostra que para um problema de 84 qubits, o Optimized Real Amplitudes requer ``population_size`` de 120, enquanto para um problema de 56 qubits, um ``population_size`` de 40 é suficiente.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Também é possível escolher um ansatz específico. O exemplo a seguir usa o ansatz ``'Tailored'``.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 4. Execute o problema.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 5. Recupere os resultados
A função retorna um dicionário com as trajetórias de investimento ordenadas do menor para o maior de acordo com o valor de sua função objetivo (veja a seção [Saída](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) da referência da API). Este conjunto de resultados permite a identificação da trajetória com o menor custo e suas correspondentes avaliações de investimento. Além disso, fornece a análise de diferentes trajetórias, facilitando a seleção das que melhor se alinham com necessidades ou objetivos específicos. Essa flexibilidade garante que as escolhas possam ser adaptadas para atender a uma variedade de preferências ou cenários.
Comece apresentando a estratégia resultante que alcançou o menor custo objetivo encontrado durante o processo.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Retrieve results

The function returns a dictionary with the investment trajectories ordered from lowest to highest according to their objective function value (see the [Output](/docs/api/functions/global-data-quantum-optimizer#output) section of the API reference). This set of results allows for the identification of the trajectory with the lowest cost and its corresponding investment evaluations. Additionally, it provides for the analysis of different trajectories, facilitating the selection of those that best align with specific needs or objectives. This flexibility ensures that choices can be tailored to suit a variety of preferences or scenarios.

Begin by presenting the resulting strategy that achieved the lowest objective cost found during the process.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Posteriormente, usando os metadados, você pode acessar os resultados de todas as estratégias amostradas. Você pode, assim, analisar mais a fundo as trajetórias alternativas retornadas pelo otimizador. Para isso, leia o dicionário armazenado em `dpo_result['metadata']['all_samples_metrics']`, que contém não apenas informações adicionais sobre a estratégia ideal, mas também detalhes das outras estratégias candidatas avaliadas durante a otimização.

O exemplo a seguir mostra como ler essas informações usando `pandas` para extrair métricas-chave associadas à estratégia ideal. Estas incluem Desvio de Restrição, Índice de Sharpe e o retorno de investimento correspondente.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
* - Visit the [API reference](/docs/api/functions/global-data-quantum-optimizer) for this Qiskit Function.
*   Request access to the function by filling in [this form](https://globaldatum.io/qiskit-function/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>